## 1. The Core Hypothesis: Bypassing SFT

Historically, post-training an LLM followed a strict two-stage paradigm:

$$
\text{Pre-trained Base Model} \longrightarrow \text{Supervised Fine-Tuning (SFT)} \longrightarrow \text{Reinforcement Learning (RLHF/RLAIF)}
$$

During SFT, the model is trained on large amounts of human-annotated step-by-step solutions. The authors of DeepSeek hypothesized that this phase may impose a performance ceiling on reasoning.

### Why this matters

- SFT locks the policy into a narrow distribution of reasoning trajectories.
- Pure RL from a base model allows the policy to explore more flexible and potentially more powerful reasoning paths.


DeepSeek-R1-Zero was trained directly on top of DeepSeek-V3-Base using pure RL without a single SFT example.


## 2. The RL Engine: Group Relative Policy Optimization (GRPO)

To make pure RL feasible across thousands of GPU hours without running out of VRAM, the paper uses GRPO.


## The GRPO Mathematical Objective

GRPO eliminates the critic model entirely. For every prompt $q$, it samples a group of $G$ outputs

$$
\{o_1, o_2, \dots, o_G\}
$$

from the old policy $\pi_{\theta_{\text{old}}}$.

The official GRPO objective is:

$$
\mathcal{J}_{\text{GRPO}}(\theta)
=
\mathbb{E}_{\substack{q \sim P(Q) \\ \{o_i\}_{i=1}^G \sim \pi_{\theta_{\text{old}}}(O\mid q)}}
\left[
\frac{1}{G}\sum_{i=1}^{G}
\left(
\min \left(
\frac{\pi_\theta(o_i \mid q)}{\pi_{\theta_{\text{old}}}(o_i \mid q)} A_i,
\; \text{clip}\left(
\frac{\pi_\theta(o_i \mid q)}{\pi_{\theta_{\text{old}}}(o_i \mid q)}, 1-\epsilon, 1+\epsilon
\right) A_i
\right)
- \beta D_{\text{KL}}(\pi_\theta \,\|\|\, \pi_{\text{ref}})
\right)
\right]
$$

The group-relative advantage is computed from the scalar rewards of that group:

$$
A_i = \frac{r_i - \text{mean}(\{r_1, r_2, \dots, r_G\})}{\text{std}(\{r_1, r_2, \dots, r_G\}) + \epsilon}
$$


## 3. Rule-Based Reward Design ($R_{\text{rule}}$)

The paper explicitly avoids neural reward models for reasoning tasks.

### Why avoid neural reward models?

When a neural reward model is trained to score complex reasoning, the policy can learn reward hacking:

- it may exploit superficial formatting tricks,
- use buzzwords,
- or rely on repetitive phrasing to inflate scores while actual accuracy collapses.

Retraining neural reward models also adds significant pipeline complexity.


The rule-based reward is defined as:

$$
R_{\text{rule}} = R_{\text{acc}} + R_{\text{format}}
$$


## Reference Model Update Schedule

The reference policy $\pi_{\text{ref}}$ is updated every $400$ steps to the latest policy $\pi_\theta$, allowing the model to keep exploring without being overly constrained by KL penalties.
